In [0]:
from pyspark.sql.functions import col, lag, round, when, date_add, current_timestamp
from pyspark.sql.window import Window

In [0]:
src_Table='project_mobility.bronze.fuel_price_raw'
target_Table='project_mobility.silver.fuel_prices_clean'

In [0]:
df = spark.read.table(src_Table)

window = Window.orderBy("week_start_date")

df_derived = df.withColumns({
      "week_end_date":   date_add(col("week_start_date"), 6),
      "price_change":    round(col("price_per_gallon") - lag("price_per_gallon", 1).over(window), 3),
      "ingestion_timestamp": current_timestamp()
  })

df_derived = df_derived.withColumn(
      "price_trend", when(col("price_change") > 0,  "Increasing")
                    .when(col("price_change") < 0,  "Decreasing")
                    .when(col("price_change") == 0, "Stable")
                    .otherwise("Stable")
  )

df_derived.write.mode("overwrite").saveAsTable(target_Table)

#display(df_derived).orderBy("week_start_date")